# PPO DOJO — kit1 retrain (GPU)

1. **Runtime → Change runtime type → T4 GPU** first (else this trains on CPU).
2. Edit `REPO` below to your repo URL, then **Runtime → Run all**.
3. Training runs in the background; the dashboard cell opens a live tab.

In [ ]:
REPO = "https://github.com/YOU/ppo-dojo.git"  # <-- edit me
ITERS = 1000
ENVS = 32
TICKS = 256
OUTDIR = "runs/kit1"
RESET_POOL = True  # drop coward league snapshots from the checkpoint

In [ ]:
!git clone $REPO dojo
!pip install -q numpy  # torch ships with CUDA on colab; do NOT pip install torch

In [ ]:
!nvidia-smi -L || echo 'NO GPU VISIBLE - set Runtime -> Change runtime type -> T4 GPU'
!cd dojo && python -c "import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"

In [ ]:
!cd dojo && python -m mc_pvp.validate 2>&1 | tail -2

In [ ]:
# training runs in the BACKGROUND so later cells (dashboard) can run.
# logs go to dojo/train_colab.log; checkpoints still save every 25 iters.
cmd = (f"cd dojo && nohup python -u train.py --iters {ITERS} --envs {ENVS} --ticks {TICKS} "
       f"--outdir {OUTDIR} --resume" + (" --reset-pool" if RESET_POOL else "") +
       f" > train_colab.log 2>&1 & echo started pid $!")
print("+", cmd)
!{cmd}

In [ ]:
# peek at training progress anytime (re-run this cell)
!tail -n 8 dojo/train_colab.log

In [ ]:
# live dashboard: served from inside colab, embedded below.
# (no query string: the page defaults to runs/kit1 on its own)
!cd dojo && (pgrep -f "http.server 8000" || nohup python -m http.server 8000 >/dev/null 2>&1 &)
!sleep 1; curl -s -o /dev/null -w "local server: %{http_code} (want 200)\n" http://localhost:8000/tools/dashboard.html
from google.colab import output
output.serve_kernel_port_as_iframe(8000, path='/tools/dashboard.html', width=1100, height=900)

In [ ]:
# when training is done: pull the weights back to your PC
from google.colab import files
files.download('dojo/runs/kit1/ppo.pt')